In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import (
    roc_auc_score, roc_curve, classification_report, confusion_matrix,
    precision_score, recall_score, f1_score, accuracy_score, average_precision_score,
    precision_recall_curve
)
import xgboost as xgb
import warnings
import sys
import joblib
import os
warnings.filterwarnings('ignore')

# Add src to path
sys.path.append('../../')
from src.data.processors.data_cleaning import (
    filter_shot_events, flatten_nested_columns, remove_unnecessary_columns,
    standardize_coordinates, handle_missing_data, create_goal_target
)
from src.features.feature_engineering import (
    calculate_shot_distance, calculate_shot_angle, extract_game_situation, time_to_seconds
)

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

In [8]:
print("="*60)
print("1. LOADING RAW DATA")
print("="*60)

# Load RAW data to demonstrate the start-to-finish pipeline
INPUT_FILE = "../../data/raw/nhl_raw_plays.parquet"
df_raw = pd.read_parquet(INPUT_FILE)

print(f"Raw Data shape: {df_raw.shape}")
print(f"Total events: {len(df_raw):,}")

# Filter for shots and create target (since sklearn Pipeline transformations 
# must preserve rows to align with target variable during .fit, we split out target before pipeline)
print("\nFiltering for shot events to prepare target variable...")
shot_events = ['shot-on-goal', 'missed-shot', 'blocked-shot', 'goal']
df_shots = filter_shot_events(df_raw, shot_events)
df_shots = create_goal_target(df_shots)

# Separate predictors and target
X = df_shots.drop(columns=['is_goal'])
y = df_shots['is_goal']

print(f"Total shots for modeling: {len(X):,}")
print(f"Total goals: {y.sum():,}")
print(f"Goal rate: {y.mean():.2%}")

1. LOADING RAW DATA
Raw Data shape: (2638947, 11)
Total events: 2,638,947

Filtering for shot events to prepare target variable...
Total shots for modeling: 985,008
Total goals: 53,037
Goal rate: 5.38%


In [9]:
print("="*60)
print("2. DEFINING THE PIPELINE")
print("="*60)

# Create wrappers for our data cleaning and feature engineering functions
# so they can be chained in a scikit-learn Pipeline.

def clean_data(X):
    X = flatten_nested_columns(X)
    X = remove_unnecessary_columns(X)
    X = standardize_coordinates(X)
    X = handle_missing_data(X)
    return X

def engineer_features(X):
    X = X.copy()
    
    # Basic features
    X['distance'] = calculate_shot_distance(X)
    X['angle'] = calculate_shot_angle(X)
    
    # Game situation features
    X['game_situation'] = X['situationCode'].apply(extract_game_situation)
    X['is_even_strength'] = (X['game_situation'] == 'even_strength').astype(int)
    X['is_empty_net'] = (X['game_situation'] == 'empty_net').astype(int)
    X['is_man_advantage'] = (X['game_situation'] == 'man_advantage').astype(int)
    
    # Time features
    X['time_in_period_seconds'] = X['timeInPeriod'].apply(time_to_seconds)
    X['time_remaining_seconds'] = X['timeRemaining'].apply(time_to_seconds)
    X['time_remaining_game'] = (X['period'] - 1) * 1200 + X['time_remaining_seconds'].fillna(0)
    
    X['is_regulation'] = (X['periodType'] == 'REG').astype(int)
    X['is_overtime'] = (X['periodType'] == 'OT').astype(int)
    X['is_shootout'] = (X['periodType'] == 'SO').astype(int)
    
    X['is_late_game'] = ((X['period'] <= 3) & (X['time_remaining_seconds'] <= 300)).astype(int)
    X['is_early_period'] = (X['time_in_period_seconds'] <= 120).astype(int)
    
    # Shot quality metrics
    X['distance_angle_interaction'] = X['distance'] * X['angle'].abs()
    X['normalized_distance'] = X['distance'] / 100
    X['normalized_angle'] = X['angle'].abs() / 90
    X['shot_quality_score'] = X['normalized_distance'] * 0.6 + X['normalized_angle'] * 0.4
    
    X['is_high_danger'] = ((X['distance'] < 25) & (X['angle'].abs() < 30)).astype(int)
    X['is_medium_danger'] = ((X['distance'] < 40) & (X['angle'].abs() < 45) & ~X['is_high_danger']).astype(int)
    X['is_low_danger'] = (~X['is_high_danger'] & ~X['is_medium_danger']).astype(int)
    
    # Rebound features 
    X_sorted = X.sort_values('eventId').reset_index(drop=False)
    X_sorted['prev_event_id'] = X_sorted['eventId'].shift(1)
    X_sorted['event_id_diff'] = X_sorted['eventId'] - X_sorted['prev_event_id']
    X_sorted['is_potential_rebound'] = ((X_sorted['event_id_diff'] <= 5) & (X_sorted['event_id_diff'] > 0)).astype(int)
    
    # Merge back to original using index to maintain order of X
    X['is_potential_rebound'] = X_sorted.set_index('index')['is_potential_rebound'].fillna(0).astype(int)
    
    # Shot types
    common_shot_types = ['wrist', 'snap', 'slap', 'tip-in', 'backhand', 'deflected']
    for shot_type in common_shot_types:
        X[f'is_{shot_type}'] = (X['shotType'] == shot_type).astype(int)
        
    # Zones
    X['is_offensive_zone'] = (X['zoneCode'] == 'O').astype(int)
    X['is_defensive_zone'] = (X['zoneCode'] == 'D').astype(int)
    X['is_neutral_zone'] = (X['zoneCode'] == 'N').astype(int)
    
    return X

def drop_unnecessary_features(X):
    # Exclude non-numeric and identifier columns
    exclude_cols = [
        'eventId', 'timeInPeriod', 'timeRemaining', 'situationCode', 
        'typeDescKey', 'typeCode', 'sortOrder', 'assist1PlayerId', 
        'assist1PlayerTotal', 'assist2PlayerId', 'assist2PlayerTotal',
        'awaySOG', 'awayScore', 'blockingPlayerId', 'eventOwnerTeamId', 
        'goalieInNetId', 'homeSOG', 'homeScore', 'scoringPlayerId', 
        'scoringPlayerTotal', 'shootingPlayerId', 'xCoord', 'yCoord', 
        'zoneCode', 'period', 'periodType', 'shotType', 'homeTeamDefendingSide',
        'game_situation', 'details', 'periodDescriptor'
    ]
    cols_to_drop = [c for c in exclude_cols if c in X.columns]
    X_numeric = X.drop(columns=cols_to_drop)
    
    # Any remaining object columns we just drop for simplicity
    cat_cols = X_numeric.select_dtypes(include=['object', 'category']).columns
    X_numeric = X_numeric.drop(columns=cat_cols)
    
    # Convert all columns to float/int to be safe for XGBoost
    X_numeric = X_numeric.apply(pd.to_numeric, errors='coerce').fillna(0)
    return X_numeric

# Create Scikit-learn Feature Pipeline
preprocessor = Pipeline([
    ('data_cleaning', FunctionTransformer(clean_data)),
    ('feature_engineering', FunctionTransformer(engineer_features)),
    ('feature_selection', FunctionTransformer(drop_unnecessary_features))
])

# Build total ML Pipeline with XGBoost
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='auc',
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbosity=0
    ))
])

print("\nPipeline fully defined with data cleaning, feature engineering, and XGBoost!")

2. DEFINING THE PIPELINE

Pipeline fully defined with data cleaning, feature engineering, and XGBoost!


In [10]:
print("="*60)
print("3. MODEL EVALUATION WITH CROSS-VALIDATION")
print("="*60)

# Define 3-Fold Stratified Cross Validation
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Ensure indices are reset for CV splitting
X_reset = X.reset_index(drop=True)
y_reset = y.reset_index(drop=True)

oof_preds = np.zeros(len(y_reset))
cv_auc_scores = []

print("Starting Cross-Validation... (This may take a few minutes as preprocessing runs per fold)")
for fold, (train_idx, val_idx) in enumerate(cv.split(X_reset, y_reset)):
    print(f"\nFold {fold+1}/{cv.get_n_splits()}:")
    X_train_fold, X_val_fold = X_reset.iloc[train_idx], X_reset.iloc[val_idx]
    y_train_fold, y_val_fold = y_reset.iloc[train_idx], y_reset.iloc[val_idx]
    
    # Fit the entire pipeline on the training fold
    full_pipeline.fit(X_train_fold, y_train_fold)
    
    # Predict on the validation fold
    val_preds = full_pipeline.predict_proba(X_val_fold)[:, 1]
    oof_preds[val_idx] = val_preds
    
    fold_auc = roc_auc_score(y_val_fold, val_preds)
    cv_auc_scores.append(fold_auc)
    print(f"  ✓ Fold {fold+1} AUC: {fold_auc:.4f}")

print("\n" + "-"*40)
print(f"Mean CV AUC: {np.mean(cv_auc_scores):.4f} (+/- {np.std(cv_auc_scores)*2:.4f})")
print(f"OOF Overall AUC: {roc_auc_score(y_reset, oof_preds):.4f}")
print("-"*40)

3. MODEL EVALUATION WITH CROSS-VALIDATION
Starting Cross-Validation... (This may take a few minutes as preprocessing runs per fold)

Fold 1/3:
Removed duplicate columns. New shape: (656672, 48)
Removed duplicate columns. New shape: (328336, 48)
  ✓ Fold 1 AUC: 0.8190

Fold 2/3:
Removed duplicate columns. New shape: (656672, 48)
Removed duplicate columns. New shape: (328336, 48)
  ✓ Fold 2 AUC: 0.8210

Fold 3/3:
Removed duplicate columns. New shape: (656672, 48)
Removed duplicate columns. New shape: (328336, 48)
  ✓ Fold 3 AUC: 0.8207

----------------------------------------
Mean CV AUC: 0.8202 (+/- 0.0017)
OOF Overall AUC: 0.8202
----------------------------------------


In [11]:
print("="*60)
print("4. HYPERPARAMETER TUNING - XGBOOST (PIPELINE)")
print("="*60)

# Define parameter grid for the pipeline steps
param_grid = {
    'classifier__max_depth': [4, 6],
    'classifier__learning_rate': [0.1],
    'classifier__n_estimators': [100]
}

# Grid search with cross-validation using the Pipeline
grid_search = GridSearchCV(
    full_pipeline,
    param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

# NOTE: Uncomment to run grid search on a subset of data (can be slow without subset)
# subset_idx = np.random.choice(len(X_reset), size=int(len(X_reset)*0.2), replace=False)
# print("Running GridSearchCV on a 20% sample of data...")
# grid_search.fit(X_reset.iloc[subset_idx], y_reset.iloc[subset_idx])
# print(f"Best params: {grid_search.best_params_}")
# print(f"Best AUC: {grid_search.best_score_:.4f}")
print("Skipping full GridSearchCV execution to save time, using base pipeline defaults.")

4. HYPERPARAMETER TUNING - XGBOOST (PIPELINE)
Skipping full GridSearchCV execution to save time, using base pipeline defaults.


In [12]:
print("="*60)
print("5. END-TO-END INFERENCE DEMONSTRATION")
print("="*60)

# 1. We train the final Pipeline on ALL available data
print("Training final pipeline on all data...")
full_pipeline.fit(X_reset, y_reset)

# 2. We grab a single raw play from our dataset as if it came from a live feed
# In production, this would be fresh raw nested JSON from the NHL API!
sample_play_raw = df_raw[df_raw["typeDescKey"] == "shot-on-goal"].iloc[[0]].copy()

# 3. We run the prediction using our fully trained pipeline directly on the raw data!
predicted_prob = full_pipeline.predict_proba(sample_play_raw)[0, 1]

print(f"\nSample Play Info: {sample_play_raw['typeDescKey'].values[0]}")
print(f"\nPredicted Probability of Goal (xG): {predicted_prob:.4f}")

# Save the unified pipeline securely
os.makedirs("../../data/models", exist_ok=True)
model_path = "../../data/models/nhl_xg_pipeline.pkl"
joblib.dump(full_pipeline, model_path)
print(f"\nSaved full prediction pipeline to {model_path}")


5. END-TO-END INFERENCE DEMONSTRATION
Training final pipeline on all data...
Removed duplicate columns. New shape: (985008, 48)
Removed duplicate columns. New shape: (1, 48)

Sample Play Info: shot-on-goal

Predicted Probability of Goal (xG): 0.0228

Saved full prediction pipeline to ../../data/models/nhl_xg_pipeline.pkl
